# parte 4 - aprendizagem nao supervisionada
## 4.a clusterizacao (kmeans) e 4.b reducao de dimensionalidade (pca)

neste notebook vamos aplicar tecnicas de aprendizado nao supervisionado
para encontrar grupos de casas parecidas e visualizar os dados em 2d.

In [ ]:
# importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# carregando os dados que o felipe usou
df = pd.read_csv('train.csv')

## repetindo as transformacoes que o felipe fez
precisamos das mesmas variaveis que ele criou para usar nos modelos

In [ ]:
# target encoding no bairro (igual o felipe fez)
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['Neighborhood'])
df['Neighborhood'] = encoder.fit_transform(df['Neighborhood'], df['SalePrice'])

In [ ]:
# transformando variaveis categoricas em numeros (ordinal encoding)
colunas_ordinais = ['ExterQual', 'KitchenQual', 'BsmtQual']
mapeamento = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1}

df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento)

# preenchendo valores faltantes do bsmtqual com 'po' (pobre)
df['BsmtQual'] = df['BsmtQual'].fillna('Po')
mapeamento_completo = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento_completo)

In [ ]:
# criando as mesmas features novas que o felipe criou
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
df['TotalArea'] = df['GrLivArea'] + df['TotalBsmtSF']
df['TotalBath'] = (df['FullBath'] + 0.5*df['HalfBath']
                   + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

## 4.a - clusterizacao com kmeans

vamos usar as 10 melhores features que o felipe identificou na correlacao
para agrupar as casas em grupos parecidos.

In [ ]:
# selecionando as melhores features (as que o felipe achou)
features = [
    'OverallQual', 'TotalArea', 'Neighborhood', 'ExterQual',
    'KitchenQual', 'GarageCars', 'TotalBath', 'BsmtQual',
    'HouseAge', 'YearsSinceRemodel'
]

X = df[features].copy()

# preenchendo qualquer valor nulo que tenha sobrado
X = X.fillna(0)

# padronizando os dados para o kmeans funcionar bem
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# metodo do cotovelo para encontrar o melhor k
inercia = []
silhueta = []
k_range = range(2, 7)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inercia.append(kmeans.inertia_)
    silhueta.append(silhouette_score(X_scaled, kmeans.labels_))

# grafico do cotovelo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inercia, 'bo-')
axes[0].set_xlabel('numero de clusters (k)')
axes[0].set_ylabel('inercia')
axes[0].set_title('metodo do cotovelo')
axes[0].grid(True)

axes[1].plot(k_range, silhueta, 'ro-')
axes[1].set_xlabel('numero de clusters (k)')
axes[1].set_ylabel('silhueta')
axes[1].set_title('coeficiente de silhueta')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print('valores de silhueta:')
for k, s in zip(k_range, silhueta):
    print(f'k={k}: {s:.4f}')

In [ ]:
# escolhendo k=3 (o melhor pelo grafico de silhueta)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f'quantidade de casas em cada cluster:')
print(df['cluster'].value_counts().sort_index())

In [ ]:
# usando pca para visualizar os clusters em 2d
pca_vis = PCA(n_components=2, random_state=42)
X_pca_vis = pca_vis.fit_transform(X_scaled)

df['pca1'] = X_pca_vis[:, 0]
df['pca2'] = X_pca_vis[:, 1]

# scatter plot dos clusters
plt.figure(figsize=(10, 6))
cores = ['blue', 'orange', 'green']

for i in range(k):
    mascara = df['cluster'] == i
    plt.scatter(df.loc[mascara, 'pca1'], df.loc[mascara, 'pca2'],
                c=cores[i], label=f'cluster {i}', alpha=0.6, edgecolors='k', s=50)

plt.xlabel('componente principal 1')
plt.ylabel('componente principal 2')
plt.title('clusters das casas visualizados com pca')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# perfil de cada cluster (media das features)
colunas_perfil = features + ['SalePrice']
perfil = df.groupby('cluster')[colunas_perfil].mean().round(2)

print('perfil medio de cada cluster:')
print(perfil)
print()
print('quantidade de casas por cluster:')
print(df['cluster'].value_counts().sort_index())

### interpretacao dos clusters

olhando a tabela acima da pra entender o perfil de cada grupo:

- **cluster 0**: casas com qualidade media, area media, preco medio (o grupo mais comum)
- **cluster 1**: casas mais antigas, menor qualidade, area menor, preco mais baixo
- **cluster 2**: casas mais novas, qualidade alta, area grande, preco mais alto

isso mostra que o kmeans conseguiu separar as casas em grupos que fazem sentido

## 4.b - reducao de dimensionalidade com pca

vamos usar o pca para reduzir as 10 features para 2 componentes
e ver o quanto de informacao conseguimos manter.

In [ ]:
# aplicando pca com todas as componentes
pca = PCA(random_state=42)
pca.fit(X_scaled)

# variancia explicada por cada componente
variancia_explicada = pca.explained_variance_ratio_
variancia_acumulada = np.cumsum(variancia_explicada)

print('variancia explicada por cada componente:')
for i, (v, a) in enumerate(zip(variancia_explicada, variancia_acumulada)):
    print(f'pc{i+1}: {v:.2%} | acumulado: {a:.2%}')

In [ ]:
# grafico da variancia explicada
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(variancia_explicada)+1), variancia_explicada, alpha=0.7)
plt.plot(range(1, len(variancia_explicada)+1), variancia_acumulada, 'ro-')
plt.xlabel('componente principal')
plt.ylabel('variancia explicada')
plt.title('variancia explicada por componente')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(range(1, len(variancia_explicada)+1), variancia_acumulada, alpha=0.7, color='green')
plt.axhline(y=0.8, color='r', linestyle='--', label='80%')
plt.xlabel('componente principal')
plt.ylabel('variancia acumulada')
plt.title('variancia acumulada')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# scatter plot com 2 componentes colorido pelo preco
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca2[:, 0], X_pca2[:, 1],
                      c=df['SalePrice'], cmap='viridis',
                      alpha=0.6, edgecolors='k', s=50)
plt.colorbar(scatter, label='preco de venda')
plt.xlabel('componente principal 1')
plt.ylabel('componente principal 2')
plt.title('pca - casas coloridas pelo preco de venda')
plt.grid(True)
plt.show()

In [ ]:
# pesos das features nas componentes principais (loadings)
pca_2 = PCA(n_components=2, random_state=42)
pca_2.fit(X_scaled)

loadings = pd.DataFrame(
    pca_2.components_.T,
    index=features,
    columns=['pc1', 'pc2']
)

print('pesos das features nas componentes principais:')
print(loadings.round(3))
print()
print('variancia explicada com 2 componentes:')
print(f'{sum(pca_2.explained_variance_ratio_):.2%}')

In [ ]:
# grafico dos loadings pra ver quais features sao mais importantes
plt.figure(figsize=(10, 8))

for i, feature in enumerate(features):
    plt.arrow(0, 0, loadings.loc[feature, 'pc1'], loadings.loc[feature, 'pc2'],
              head_width=0.05, head_length=0.05, alpha=0.7)
    plt.text(loadings.loc[feature, 'pc1']*1.1, loadings.loc[feature, 'pc2']*1.1,
             feature, fontsize=10)

plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('componente principal 1')
plt.ylabel('componente principal 2')
plt.title('peso das features nas componentes principais (loadings)')
plt.grid(True, alpha=0.3)
plt.xlim(-0.6, 0.6)
plt.ylim(-0.6, 0.6)
plt.show()

### interpretacao do pca

com 2 componentes principais conseguimos explicar cerca de 60% da variacao
dos dados, o que nao e muito alto mas ja da pra visualizar.

olhando o grafico de loadings da pra ver que:
- **pc1** separa casas por qualidade e tamanho (overallqual, totalarea, exterqual)
- **pc2** separa mais pela idade (houseage, yearssinceremodel)

no scatter colorido pelo preco, da pra ver que casas mais caras
ficam mais para a direita no grafico (pc1 maior), mostrando que
qualidade e tamanho realmente influenciam no valor.

### conclusao da parte 4

conseguimos identificar 3 grupos de casas com caracteristicas diferentes
usando kmeans, e o pca ajudou a visualizar como as features se relacionam
com o preco de venda.